# Pure Python ReAct Agent with Intelligent Search Trigger

This notebook demonstrates a single-agent AI system that uses the ReAct (Reason + Act) pattern.
The agent can intelligently decide when to search the web vs when to use its built-in knowledge.

**Key Features:**
- Intelligent search trigger based on temporal keywords and topics
- Integration with Gemini Flash 2.0 for reasoning
- Web search capability via Serper API
- ReAct pattern implementation (Reason → Act → Observe)

**Workshop Demo - Single Agent AI System**

## Install Dependencies

First, let's install all the required packages for our ReAct agent.

In [1]:
# Install required packages
!pip install google-generativeai requests -q

## Setup API Keys

Configure your API keys for Gemini and Serper. In Colab, use the secrets panel to store your keys securely.

In [2]:
import os
from google.colab import userdata

# Set up API keys from Colab secrets
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")

print("API keys configured successfully!")

API keys configured successfully!


## Search Trigger Intelligence Module

This is the core decision-making component that automatically identifies when a web search is necessary using pattern recognition.

In [3]:
import re
from typing import Dict, List, Any, Optional

class SearchTriggerIntelligence:
    """
    The Search Trigger Intelligence module is the system's core decision-making component.
    It automatically identifies when a web search is necessary using pattern recognition
    rather than simple keyword matching.
    """

    def __init__(self):
        # Time-sensitive keywords that indicate need for current information
        self.temporal_keywords = {
            'immediate': ['now', 'currently', 'today', 'this week', 'right now'],
            'recent': ['latest', 'recent', 'new', 'fresh', 'updated', 'current'],
            'trending': ['trending', 'popular', 'viral', 'breaking', 'hot'],
            'temporal_markers': ['2025', '2024', 'this year', 'next year'],
            'news_indicators': ['news', 'developments', 'updates', 'announcement']
        }

        # Topics that require real-time information updates
        self.current_info_topics = {
            'technology': ['ai', 'artificial intelligence', 'tech', 'software', 'app', 'startup'],
            'finance': ['market', 'stock', 'crypto', 'bitcoin', 'economy', 'price'],
            'news': ['politics', 'election', 'government', 'policy', 'war'],
            'science': ['research', 'study', 'discovery', 'breakthrough', 'covid'],
            'events': ['conference', 'pycon', 'summit', 'meeting', 'event', 'happening'],
            'weather': ['weather', 'temperature', 'forecast', 'climate']
        }

        # Question patterns that typically need search
        self.search_patterns = [
            r'where is .* happening',
            r'when is .* scheduled',
            r'what is the latest',
            r'tell me about recent',
            r'current status of',
            r'how much does .* cost',
            r'what are the reviews'
        ]

    def needs_search(self, query: str) -> bool:
        """
        Analyze if a query needs web search based on multiple factors.

        Args:
            query (str): User's input query

        Returns:
            bool: True if search is needed, False otherwise
        """
        query_lower = query.lower()

        # Check for temporal keywords
        for category, keywords in self.temporal_keywords.items():
            if any(keyword in query_lower for keyword in keywords):
                print(f"🔍 Search triggered by temporal keyword ({category})")
                return True

        # Check for current info topics
        for category, topics in self.current_info_topics.items():
            if any(topic in query_lower for topic in topics):
                print(f"🔍 Search triggered by topic category ({category})")
                return True

        # Check for search patterns using regex
        for pattern in self.search_patterns:
            if re.search(pattern, query_lower):
                print(f"🔍 Search triggered by pattern: {pattern}")
                return True

        # If no triggers found, use built-in knowledge
        print("🧠 Using built-in knowledge (no search needed)")
        return False

print("Search Trigger Intelligence module loaded!")

Search Trigger Intelligence module loaded!


## Web Search Tool

Web search functionality using Serper API for getting real-time information from Google.

In [4]:
import json
import requests

class WebSearchTool:
    """
    Web search functionality using Serper API.
    Provides structured search results from Google.
    """

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://google.serper.dev/search"

    def search(self, query: str, num_results: int = 5) -> Dict[str, Any]:
        """
        Perform web search using Serper API.

        Args:
            query (str): Search query
            num_results (int): Number of results to return

        Returns:
            Dict containing search results
        """
        try:
            payload = json.dumps({"q": query, "num": num_results})
            headers = {
                'X-API-KEY': self.api_key,
                'Content-Type': 'application/json'
            }

            response = requests.post(self.base_url, headers=headers, data=payload)
            response.raise_for_status()

            return response.json()

        except requests.exceptions.RequestException as e:
            return {"error": f"Search failed: {str(e)}"}

    def format_results(self, search_results: Dict[str, Any]) -> str:
        """
        Format search results into readable text.

        Args:
            search_results (Dict): Raw search results from API

        Returns:
            str: Formatted search results
        """
        if "error" in search_results:
            return f"Search Error: {search_results['error']}"

        if "organic" not in search_results:
            return "No search results found."

        formatted = "🔍 Web Search Results:\n\n"

        for i, result in enumerate(search_results["organic"][:5], 1):
            title = result.get("title", "No title")
            snippet = result.get("snippet", "No description")
            link = result.get("link", "No link")

            formatted += f"{i}. **{title}**\n"
            formatted += f"   {snippet}\n"
            formatted += f"   Source: {link}\n\n"

        return formatted

print("Web Search Tool loaded!")

Web Search Tool loaded!


## Gemini LLM Integration

Gemini Flash 2.0 integration for reasoning and response generation - the 'thinking' part of the ReAct pattern.

In [5]:
import google.generativeai as genai

class GeminiLLM:
    """
    Gemini Flash 2.0 integration for reasoning and response generation.
    Handles the 'thinking' part of the ReAct pattern.
    """

    def __init__(self, api_key: str):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel('gemini-2.0-flash-exp')

    def generate_response(self, prompt: str) -> str:
        """
        Generate response using Gemini model.

        Args:
            prompt (str): Input prompt for the model

        Returns:
            str: Generated response
        """
        try:
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            return f"Error generating response: {str(e)}"

print("Gemini LLM integration loaded!")

Gemini LLM integration loaded!


## Pure ReAct Agent

Main ReAct Agent that combines reasoning and action capabilities using the ReAct pattern:
1. **REASON**: Analyze the query and decide what action to take
2. **ACT**: Execute the action (search web or use knowledge)
3. **OBSERVE**: Process the results and generate final response

In [6]:
class PureReActAgent:
    """
    Main ReAct Agent that combines reasoning and action capabilities.

    The ReAct pattern works as follows:
    1. REASON: Analyze the query and decide what action to take
    2. ACT: Execute the action (search web or use knowledge)
    3. OBSERVE: Process the results and generate final response
    """

    def __init__(self):
        # Initialize components with API keys from environment
        gemini_key = os.getenv("GEMINI_API_KEY")
        serper_key = os.getenv("SERPER_API_KEY")

        if not gemini_key or not serper_key:
            raise ValueError("Missing API keys in environment variables")

        self.llm = GeminiLLM(gemini_key)
        self.search_tool = WebSearchTool(serper_key)
        self.search_trigger = SearchTriggerIntelligence()

        print("ReAct Agent initialized successfully!")
        print("Ready to reason and act on your queries!")

    def reason(self, query: str) -> Dict[str, Any]:
        """
        REASON step: Analyze the query and decide on action.

        Args:
            query (str): User's input query

        Returns:
            Dict containing reasoning results
        """
        print(f"\n REASONING about: '{query}'")

        # Use search trigger intelligence to decide
        needs_search = self.search_trigger.needs_search(query)

        reasoning = {
            "query": query,
            "needs_search": needs_search,
            "action": "web_search" if needs_search else "direct_response",
            "reasoning": f"Query {'requires' if needs_search else 'does not require'} web search"
        }

        print(f"💭 Decision: {reasoning['action']}")
        return reasoning

    def act(self, reasoning: Dict[str, Any]) -> Dict[str, Any]:
        """
        ACT step: Execute the decided action.

        Args:
            reasoning (Dict): Results from reasoning step

        Returns:
            Dict containing action results
        """
        query = reasoning["query"]
        action = reasoning["action"]

        print(f"\n⚡ ACTING: {action}")

        if action == "web_search":
            # Perform web search
            search_results = self.search_tool.search(query)
            formatted_results = self.search_tool.format_results(search_results)

            return {
                "action": action,
                "results": formatted_results,
                "raw_data": search_results
            }

        else:
            # Use direct LLM response
            prompt = f"""
            Answer the following question using your built-in knowledge:

            Question: {query}

            Provide a helpful and informative response.
            """

            response = self.llm.generate_response(prompt)

            return {
                "action": action,
                "results": response,
                "raw_data": None
            }

    def observe_and_respond(self, query: str, action_results: Dict[str, Any]) -> str:
        """
        OBSERVE step: Process action results and generate final response.

        Args:
            query (str): Original user query
            action_results (Dict): Results from action step

        Returns:
            str: Final response to user
        """
        print(f"\n OBSERVING results and generating response...")

        if action_results["action"] == "web_search":
            # Synthesize search results into coherent response
            prompt = f"""
            Based on the following web search results, provide a comprehensive answer to the user's question.

            User Question: {query}

            Search Results:
            {action_results["results"]}

            Instructions:
            - Synthesize the information from multiple sources
            - Provide specific details and facts
            - Mention sources when relevant
            - Keep the response informative but concise
            """

            final_response = self.llm.generate_response(prompt)

        else:
            # Direct response from LLM
            final_response = action_results["results"]

        return final_response

    def process_query(self, query: str) -> str:
        """
        Main method that implements the complete ReAct cycle.

        Args:
            query (str): User's input query

        Returns:
            str: Final response
        """
        print("="*60)
        print(" Starting ReAct Process")
        print("="*60)

        try:
            # Step 1: REASON
            reasoning = self.reason(query)

            # Step 2: ACT
            action_results = self.act(reasoning)

            # Step 3: OBSERVE and respond
            final_response = self.observe_and_respond(query, action_results)

            print(f"\n FINAL RESPONSE:")
            print("-" * 40)
            return final_response

        except Exception as e:
            error_msg = f" Error in ReAct process: {str(e)}"
            print(error_msg)
            return error_msg

print(" Pure ReAct Agent class loaded!")

 Pure ReAct Agent class loaded!


## Initialize and Test the Agent

Let's create our ReAct agent and test it with some example queries.

In [7]:
# Initialize the ReAct agent
agent = PureReActAgent()

ReAct Agent initialized successfully!
Ready to reason and act on your queries!


## Example Queries

Let's test our agent with different types of queries to see how it decides between web search and built-in knowledge.

In [8]:
# Example 1: Technical question
query1 = "Write a FastAPI backend for calling a weather API"
response1 = agent.process_query(query1)
print(response1)

 Starting ReAct Process

 REASONING about: 'Write a FastAPI backend for calling a weather API'
🔍 Search triggered by topic category (weather)
💭 Decision: web_search

⚡ ACTING: web_search

 OBSERVING results and generating response...

 FINAL RESPONSE:
----------------------------------------
Here's a comprehensive guide to creating a FastAPI backend for calling a weather API, based on the provided search results:

**Core Functionality**

The primary goal is to build an API endpoint using FastAPI that fetches and returns weather data.  This involves:

*   **Defining API Endpoints:** You'll need to define routes (e.g., `/weather`) that clients can access to request weather information. (Source: [https://ktyptorio.medium.com/simple-openweather-api-service-using-fastapi-and-minio-object-storage-docker-version-f3587f7eb3de](https://ktyptorio.medium.com/simple-openweather-api-service-using-fastapi-and-minio-object-storage-docker-version-f3587f7eb3de))
*   **Calling an External Weather API:**

In [10]:
# Example 2: Current event question (should trigger web search)
query2 = "Where is PyCon 2025 India happening and tell me more?"
response2 = agent.process_query(query2)
print(response2)

 Starting ReAct Process

 REASONING about: 'Where is PyCon 2025 India happening and tell me more?'
🔍 Search triggered by temporal keyword (temporal_markers)
💭 Decision: web_search

⚡ ACTING: web_search

 OBSERVING results and generating response...

 FINAL RESPONSE:
----------------------------------------
PyCon India 2025 will be held in **Bengaluru, India** from **September 12th to 15th, 2025** (Sources: in.pycon.org, python.org, konfhub.com).

Specifically:

*   **September 12th** will be dedicated to workshops at St. Francis College, Bengaluru (in.pycon.org).
*   **September 13th and 14th** will feature the main conference at NIMHANS Convention Centre, Bengaluru (in.pycon.org).
*   **September 15th** will be for development sprints (Dev).

The venue is centrally located in Bengaluru and has two floors, with the ground floor hosting the main tracks in three halls (in.pycon.org/2025/venue/). The event is organized entirely by volunteers (konfhub.com).



In [11]:
# Example 3: Latest developments (should trigger web search)
query3 = "What are the latest developments in AI?"
response3 = agent.process_query(query3)
print(response3)

 Starting ReAct Process

 REASONING about: 'What are the latest developments in AI?'
🔍 Search triggered by temporal keyword (recent)
💭 Decision: web_search

⚡ ACTING: web_search

 OBSERVING results and generating response...

 FINAL RESPONSE:
----------------------------------------
Here's a summary of the latest developments in AI, based on the provided search results:

*   **Deepfake Detection:** A new universal AI detector has been developed that boasts 98% accuracy in identifying deepfake videos (crescendo.ai).

*   **AI in Banking:** Malaysia has launched Ryt Bank, which is its first AI-powered bank (artificialintelligence-news.com).

*   **AI Video Generation:** Google's Veo 3 AI video creation tools are now widely available (artificialintelligence-news.com). Additionally, text-to-video AI is advancing, with new "metamorphic video" capabilities (sciencedaily.com).

*   **Quantum Computing:** There's development happening in characterizing quantum gate errors, which is relevant to

## Summary

This Pure Python ReAct Agent demonstrates:

1. **Intelligent Decision Making**: Uses pattern recognition to decide when web search is needed
2. **ReAct Pattern**: Implements Reason → Act → Observe cycle
3. **Multi-Modal Responses**: Can both search the web and use built-in knowledge
4. **Temporal Awareness**: Recognizes time-sensitive queries
5. **Topic Classification**: Identifies domains that require real-time information

**Key Components:**
- `SearchTriggerIntelligence`: Decides when to search vs use knowledge
- `WebSearchTool`: Handles web search via Serper API
- `GeminiLLM`: Provides reasoning and response generation
- `PureReActAgent`: Orchestrates the complete ReAct workflow

This approach allows for a more intelligent and context-aware AI agent that can handle both factual questions and current events effectively.